# Montandon: Floods

This notebook fetches recent flood data from Montandon, the global crisis data bank, filtering for severity, and visualizes them on a map.

In [19]:
import os
import pandas as pd
from pystac_client import Client
import geopandas as gpd
from shapely.geometry import Point, Polygon, shape
from lonboard import viz

from datetime import datetime, timedelta, timezone

## Connect to Montandon STAC

Montandon is exposed as a STAC collection, but requires authentication.

In [20]:
STAC_API_URL = "https://montandon-eoapi.ifrc.org/stac"
API_TOKEN = os.getenv('MONTANDON_API_TOKEN')

In [21]:
auth_headers = {"Authorization": f"Bearer {API_TOKEN}"}

### Check Auth

In [22]:
# Connect to STAC API with authentication
try:
    client = Client.open(STAC_API_URL, headers=auth_headers)
    print(f"\n[OK] Connected to: {STAC_API_URL}")
    print(f"[OK] API Title: {client.title}")
    print(f"[OK] Authentication: Bearer Token (OpenID Connect)")
except Exception as e:
    print(f"\n[ERROR] Authentication failed: {e}")


[OK] Connected to: https://montandon-eoapi.ifrc.org/stac
[OK] API Title: Montandon STAC API
[OK] Authentication: Bearer Token (OpenID Connect)


In [23]:
client

<Client id=montandon-eoapi>

## Fetch Recent Flood Impacts

We search for flood impact records directly, and then fetch hazard data based on the monty correlation ID.

Floods are searched for by the UNDRR hazard type `MH0600`, [which are mapped](https://github.com/IFRCGo/monty-stac-extension/tree/main/docs/model/sources/GDACS#mapping-from-gdacs-event-type-to-hazard-profile) to GDACS flood event `FL`. 

In [24]:
now = datetime.now(timezone.utc)
start = now - timedelta(days=90)  # adjust to change the search window
items_max = 5000 # Keeps max items consistent across searches

_Note that this search takes several minutes to run with `items_max = 5000`._

In [25]:
search = client.search(
    collections=["gdacs-impacts"],
    datetime=f"{start.isoformat()}/{now.isoformat()}",
    max_items=items_max,
)

# MH0600 is the standard UNDRR flood hazard code
flood_impacts = [
    item for item in search.items()
    if 'MH0600' in item.properties.get('monty:hazard_codes', [])
]
print(f"Found {len(flood_impacts)} flood impact records")

Found 4678 flood impact records


Each impact below returns a row for impact type and its value.

In [26]:
def parse_impact(item):
    detail = item.properties.get('monty:impact_detail', {})
    return {
        'monty_corr_id': item.properties.get('monty:corr_id'),
        'title': item.properties.get('title'),
        'country_codes': ', '.join(item.properties.get('monty:country_codes', [])),
        'impact_type': detail.get('type'),
        'impact_value': detail.get('value'),
    }

impacts_df = pd.DataFrame([parse_impact(item) for item in flood_impacts])
impacts_df

,monty_corr_id,title,country_codes,impact_type,impact_value
0,20260519-USA-1161474-MH0600-1-GCDB,Flood in United States,USA,assisted,3
1,20260518-AUS-556064-MH0600-1-GCDB,Flood in Australia,AUS,assisted,55
2,20260518-AUS-556064-MH0600-1-GCDB,Flood in Australia,AUS,relocated,48
3,20260513-PHL-874528-MH0600-2-GCDB,Flood in Philippines,PHL,destroyed,1
4,20260513-PHL-874528-MH0600-2-GCDB,Flood in Philippines,PHL,relocated,358
...,...,...,...,...,...
4673,20260302-IDN-816913-MH0600-4-GCDB,Flood in Indonesia,IDN,relocated,111
4674,20260302-IDN-816913-MH0600-4-GCDB,Flood in Indonesia,IDN,affected_total,435
4675,20260302-IDN-816913-MH0600-49-GCDB,Flood in Indonesia,IDN,damaged,131
4676,20260302-IDN-816913-MH0600-49-GCDB,Flood in Indonesia,IDN,relocated,111


We pivot the rows, so that each event, matched with the Monty correlation id `monty_corr_id`, is a single row, with columns being the values of potential impacts.

In [27]:
# Pivot: one row per event, impact types as columns
impacts_pivot = impacts_df.pivot_table(
    index='monty_corr_id',
    columns='impact_type',
    values='impact_value',
    aggfunc='sum'
).reset_index()

# Re-attach metadata — take first value per corr_id
metadata = (
    impacts_df[['monty_corr_id', 'title', 'country_codes']]
    .drop_duplicates('monty_corr_id')
)
impacts_pivot = impacts_pivot.merge(metadata, on='monty_corr_id')

print(f"{len(impacts_pivot)} flood events with impact data")
impacts_pivot

91 flood events with impact data


,monty_corr_id,affected_total,assisted,damaged,death,destroyed,injured,missing,relocated,title,country_codes
0,20260302-IDN-779109-MH0600-38-GCDB,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,Flood in Indonesia,IDN
1,20260302-IDN-779109-MH0600-39-GCDB,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,Flood in Indonesia,IDN
2,20260302-IDN-779109-MH0600-40-GCDB,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,Flood in Indonesia,IDN
3,20260302-IDN-779109-MH0600-41-GCDB,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,Flood in Indonesia,IDN
4,20260302-IDN-779109-MH0600-42-GCDB,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,Flood in Indonesia,IDN
...,...,...,...,...,...,...,...,...,...,...,...
86,20260515-CHN-1077856-MH0600-3-GCDB,NaN,5.0,50.0,7.0,NaN,NaN,13.0,6.0,Flood in China,CHN
87,20260515-CHN-1077856-MH0600-4-GCDB,NaN,5.0,50.0,8.0,NaN,NaN,15.0,18542.0,Flood in China,CHN
88,20260516-MYS-824018-MH0600-1-GCDB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,341.0,Flood in Malaysia,MYS
89,20260518-AUS-556064-MH0600-1-GCDB,NaN,55.0,NaN,NaN,NaN,NaN,NaN,48.0,Flood in Australia,AUS


## Get Flood Footprints

Use the correlation IDs from the impact records to fetch the matching flood footprint polygons from GDACS hazards.

In [28]:
corr_id_set = set(impacts_pivot['monty_corr_id'])

search = client.search(
    collections=["gdacs-hazards"],
    max_items=items_max,
)

hazard_items = [
    item for item in search.items()
    if item.properties.get('monty:corr_id') in corr_id_set
]
print(f"Matched {len(hazard_items)} hazard footprints")

Matched 85 hazard footprints


In [29]:
hazards_gdf = gpd.GeoDataFrame(
    [{
        'monty_corr_id': item.properties['monty:corr_id'],
        'severity_label': item.properties.get('monty:hazard_detail', {}).get('severity_label'),
        'geometry': shape(item.geometry),
    } for item in hazard_items],
    geometry='geometry',
    crs='EPSG:4326',
)

# Combine footprints with impact data
floods = hazards_gdf.merge(impacts_pivot, on='monty_corr_id', how='left')
floods

,monty_corr_id,severity_label,geometry,affected_total,assisted,damaged,death,destroyed,injured,missing,relocated,title,country_codes
0,20260519-USA-1161474-MH0600-1-GCDB,Green,"POLYGON ((-85.4007 39.0812, -85.3681 39.0678, ...",NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,Flood in United States,USA
1,20260518-AUS-556064-MH0600-1-GCDB,Green,"POLYGON ((152.7325 -28.2175, 152.7022 -28.3378...",NaN,55.0,NaN,NaN,NaN,NaN,NaN,48.0,Flood in Australia,AUS
2,20260516-MYS-824018-MH0600-1-GCDB,Green,"POLYGON ((103.3902 1.6221, 103.4803 1.6221, 10...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,341.0,Flood in Malaysia,MYS
3,20260515-CHN-1077856-MH0600-4-GCDB,Green,"POLYGON ((111.3893 29.2735, 111.5378 29.5887, ...",NaN,5.0,50.0,8.0,NaN,NaN,15.0,18542.0,Flood in China,CHN
4,20260515-CHN-1077856-MH0600-3-GCDB,Green,"POLYGON ((108.1457 24.978, 108.2358 24.9758, 1...",NaN,5.0,50.0,7.0,NaN,NaN,13.0,6.0,Flood in China,CHN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,20260302-IDN-816913-MH0600-14-GCDB,Green,"POLYGON ((106.8194 -6.3712, 106.8324 -6.3673, ...",13320.0,NaN,4374.0,10.0,NaN,9.0,3.0,625.0,Flood in Indonesia,IDN
81,20260302-IDN-816913-MH0600-13-GCDB,Green,"POLYGON ((114.9325 -9.0301, 114.4238 -8.3179, ...",12635.0,NaN,3683.0,10.0,NaN,9.0,3.0,608.0,Flood in Indonesia,IDN
82,20260302-IDN-816913-MH0600-12-GCDB,Green,"POLYGON ((108.5559 -7.2886, 108.7235 -7.4315, ...",12608.0,NaN,3660.0,9.0,NaN,9.0,3.0,608.0,Flood in Indonesia,IDN
83,20260302-IDN-816913-MH0600-11-GCDB,Green,"POLYGON ((122.89 -11.2086, 124.6041 -10.2817, ...",12603.0,NaN,3660.0,9.0,NaN,9.0,3.0,608.0,Flood in Indonesia,IDN


In [30]:
viz(floods)